# FAF5 2024 – Scenario OD Line GeoJSON (Fixed)



In [2]:

# 1) Imports & Global Config
import pandas as pd
import numpy as np
from pathlib import Path

import geopandas as gpd
from shapely.geometry import LineString

PREDICTIONS_CSV = Path("FAF5_2024_predictions_GOLD.csv")
BASE_2024_CSV   = Path("FAF5_2024_ML.csv")

DOMESTIC_META_CSV = Path("FAF5_metadata_domestic.csv")
FOREIGN_META_CSV  = Path("FAF5_metadata_foreign.csv")

METRO_FAF_SHP  = Path("2017_CFS_Metro_Areas_with_FAF")
OUTPUT_GEOJSON = Path("faf5_2024_scenarios_10k_OD_lines.geojson")

N_OUT = 10_000


In [3]:

# 2) Load data 

# --- Load base ---
try:
    base_2024 = pd.read_csv(BASE_2024_CSV)
    print("Base 2024 shape:", base_2024.shape)
except FileNotFoundError:
    print("Warning: BASE_2024_CSV not found. Continuing without it.")
    base_2024 = None

# --- Load predictions ---
df = pd.read_csv(PREDICTIONS_CSV)
print("Predictions shape:", df.shape)

# --- Load metadata ---
dom_meta = pd.read_csv(DOMESTIC_META_CSV)
for_meta = pd.read_csv(FOREIGN_META_CSV)
print("Domestic metadata shape:", dom_meta.shape)
print("Foreign metadata shape:", for_meta.shape)

# --- Load metro shapefile ---
metro_gdf = gpd.read_file(METRO_FAF_SHP)
print("Metro–FAF GeoDataFrame shape:", metro_gdf.shape)

# --- Add stable id AFTER loading ---
if "id" not in df.columns:
    df = df.reset_index(drop=True)
    df["id"] = df.index.astype("int64")

if base_2024 is not None and "id" not in base_2024.columns:
    base_2024 = base_2024.reset_index(drop=True)
    base_2024["id"] = base_2024.index.astype("int64")

print("df id sample:", df["id"].head().tolist())
if base_2024 is not None:
    print("base_2024 id sample:", base_2024["id"].head().tolist())
    n_overlap = base_2024["id"].isin(df["id"]).sum()
    print("ID overlap (base_2024 ids found in df):", int(n_overlap), "out of", len(base_2024))
    if n_overlap == 0:
        raise ValueError(
            "No overlapping IDs between base_2024 and df. "
            "Carry an id/row_id through the prediction pipeline, do not recreate it later."
        )


Base 2024 shape: (1688271, 25)
Predictions shape: (1678541, 39)
Domestic metadata shape: (132, 3)
Foreign metadata shape: (8, 2)


C:\Users\id145\anaconda3\Lib\site-packages\pyogrio\core.py:35: RuntimeWarning: Could not detect GDAL data files.  Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()


Metro–FAF GeoDataFrame shape: (132, 13)
df id sample: [0, 1, 2, 3, 4]
base_2024 id sample: [0, 1, 2, 3, 4]
ID overlap (base_2024 ids found in df): 1678541 out of 1688271


In [4]:

# 2.1) Quick sanity: show prediction columns
pred_like = [c for c in df.columns if ("pred" in c.lower() or "kan" in c.lower() or "rf" in c.lower())]
print("Pred-like columns found:", pred_like)

print("First 40 columns in df:", df.columns.tolist()[:40])
if base_2024 is not None:
    print("First 40 columns in base_2024:", base_2024.columns.tolist()[:40])


Pred-like columns found: ['mode_pred', 'cost_per_tkm_pred_rf', 'co2_kg_per_tkm_pred_rf', 'cost_per_tkm_pred_kan', 'co2_kg_per_tkm_pred_kan', 'cost_pred', 'co2_pred_tonnes', 'gen_cost_pred', 'cost_pred_kan', 'co2_pred_tonnes_kan', 'gen_cost_pred_kan']
First 40 columns in df: ['dms_orig', 'dms_dest', 'dms_mode', 'mode_name', 'sctg2', 'trade_type', 'dist_band', 'tons', 'value_2024_musd', 'value_per_ton', 'ton_miles', 'distance_miles', 'distance_km', 'cost_per_tkm', 'cost_estimate', 'kg_co2_per_tkm', 'co2_kg', 'co2_tonnes', 'gen_cost_estimate', 'fuel_multiplier', 'distance_multiplier', 'scale_multiplier', 'congestion_multiplier', 'market_noise', 'emissions_noise', 'row_id', 'mode_pred', 'ton_km', 'co2_kg_per_tkm', 'cost_per_tkm_pred_rf', 'co2_kg_per_tkm_pred_rf', 'cost_per_tkm_pred_kan', 'co2_kg_per_tkm_pred_kan', 'cost_pred', 'co2_pred_tonnes', 'gen_cost_pred', 'cost_pred_kan', 'co2_pred_tonnes_kan', 'gen_cost_pred_kan', 'id']
First 40 columns in base_2024: ['dms_orig', 'dms_dest', 'dms_m

In [5]:

# 3) Merge base + predictions on id (keeps all prediction columns, restores any missing baselines)

def coalesce_cols(frame, out_name, candidates):
    cols = [c for c in candidates if c in frame.columns]
    if not cols:
        return frame
    s = frame[cols[0]]
    for c in cols[1:]:
        s = s.combine_first(frame[c])
    frame[out_name] = s
    return frame

if base_2024 is not None:
   
    if df.duplicated(["id"]).any():
        num_cols = [c for c in df.columns if c != "id"]
        df = df.groupby("id", as_index=False)[num_cols].mean(numeric_only=True)

    df = base_2024.merge(df, on="id", how="left", suffixes=("_base", "_pred"), validate="1:1")
    print("Merged df shape:", df.shape)

    # Standardize key columns so later cells never KeyError
    df = coalesce_cols(df, "cost_estimate", ["cost_estimate_pred", "cost_estimate_base", "cost_estimate"])
    df = coalesce_cols(df, "co2_tonnes",   ["co2_tonnes_pred",   "co2_tonnes_base",   "co2_tonnes"])
    df = coalesce_cols(df, "ton_km",       ["ton_km_pred",       "ton_km_base",       "ton_km"])
    df = coalesce_cols(df, "ton_miles",    ["ton_miles_pred",    "ton_miles_base",    "ton_miles"])
    df = coalesce_cols(df, "dms_orig",     ["dms_orig_pred",     "dms_orig_base",     "dms_orig"])
    df = coalesce_cols(df, "dms_dest",     ["dms_dest_pred",     "dms_dest_base",     "dms_dest"])
    df = coalesce_cols(df, "dms_mode",     ["dms_mode_pred",     "dms_mode_base",     "dms_mode"])
    df = coalesce_cols(df, "mode_name",    ["mode_name_pred",    "mode_name_base",    "mode_name"])
    df = coalesce_cols(df, "sctg2",        ["sctg2_pred",        "sctg2_base",        "sctg2"])
    df = coalesce_cols(df, "trade_type",   ["trade_type_pred",   "trade_type_base",   "trade_type"])
    df = coalesce_cols(df, "dist_band",    ["dist_band_pred",    "dist_band_base",    "dist_band"])
    df = coalesce_cols(df, "tons",         ["tons_pred",         "tons_base",         "tons"])
    df = coalesce_cols(df, "distance_miles", ["distance_miles_pred", "distance_miles_base", "distance_miles"])
    df = coalesce_cols(df, "value_2024_musd", ["value_2024_musd_pred", "value_2024_musd_base", "value_2024_musd"])

df["cost_estimate"] = pd.to_numeric(df["cost_estimate"], errors="coerce")
df["co2_tonnes"]    = pd.to_numeric(df["co2_tonnes"], errors="coerce")

print("✅ Baseline columns present:", {"cost_estimate","co2_tonnes"}.issubset(df.columns))


Merged df shape: (1688271, 65)
✅ Baseline columns present: True


In [6]:

# 4) Ensure totals exist and are consistent (RF + KAN)
def first_col(frame, cands):
    return next((c for c in cands if c in frame.columns), None)

activity_col = first_col(df, ["ton_km", "ton_miles"])
rf_cost_int  = first_col(df, ["cost_per_tkm_pred_rf"])
rf_co2_int   = first_col(df, ["co2_kg_per_tkm_pred_rf", "co2_per_tkm_pred_rf"])
kan_cost_int = first_col(df, ["cost_per_tkm_pred_kan"])
kan_co2_int  = first_col(df, ["co2_kg_per_tkm_pred_kan", "co2_per_tkm_pred_kan"])

def to_num(s): 
    return pd.to_numeric(s, errors="coerce")

if activity_col is not None:
    A = to_num(df[activity_col])

    def co2_total_from_int(int_col):
        x = to_num(df[int_col])
        if "kg" in int_col.lower():
            x = x / 1000.0
        return x * A

    if "cost_pred" not in df.columns and rf_cost_int:
        df["cost_pred"] = to_num(df[rf_cost_int]) * A
    if "co2_pred_tonnes" not in df.columns and rf_co2_int:
        df["co2_pred_tonnes"] = co2_total_from_int(rf_co2_int)

    if "cost_pred_kan" not in df.columns and kan_cost_int:
        df["cost_pred_kan"] = to_num(df[kan_cost_int]) * A
    if "co2_pred_tonnes_kan" not in df.columns and kan_co2_int:
        df["co2_pred_tonnes_kan"] = co2_total_from_int(kan_co2_int)

if "gen_cost_pred" not in df.columns and {"cost_pred","co2_pred_tonnes"}.issubset(df.columns):
    df["gen_cost_pred"] = df["cost_pred"] + df["co2_pred_tonnes"]

must_have = ["cost_pred","co2_pred_tonnes","cost_pred_kan","co2_pred_tonnes_kan","gen_cost_pred"]
missing = [c for c in must_have if c not in df.columns]
if missing:
    raise ValueError(f"Missing required prediction totals after patch: {missing}")

print("✅ Prediction totals OK.")
df[["cost_pred","co2_pred_tonnes","cost_pred_kan","co2_pred_tonnes_kan","gen_cost_pred"]].head()


✅ Prediction totals OK.


,cost_pred,co2_pred_tonnes,cost_pred_kan,co2_pred_tonnes_kan,gen_cost_pred
0,621105.80,389.47293,1443225.00,511.52524,660053.10
1,9877032.00,5778.98050,19827696.00,7365.56150,10454930.00
2,109627.94,56.59841,207137.58,73.36229,115287.78
3,436322.84,269.93225,1017423.94,358.21756,463316.06
4,245064.75,151.37021,593808.75,209.48473,260201.77


In [7]:

# 5) Attach FAF Zone Names (origin/destination)

dom_meta2 = dom_meta.rename(
    columns={
        "Numeric Label": "zone_id",
        "Short Description": "zone_name_short",
        "Long Description": "zone_name_long",
    }
)
for_meta2 = for_meta.rename(
    columns={
        "Numeric Label": "zone_id",
        "Description": "zone_name_short",
    }
)

zone_lookup = pd.concat(
    [
        dom_meta2[["zone_id", "zone_name_short"]],
        for_meta2[["zone_id", "zone_name_short"]],
    ],
    ignore_index=True,
)
zone_lookup["zone_id"] = pd.to_numeric(zone_lookup["zone_id"], errors="coerce")

df["dms_orig"] = pd.to_numeric(df["dms_orig"], errors="coerce")
df["dms_dest"] = pd.to_numeric(df["dms_dest"], errors="coerce")

# Drop prior zone-name cols 
df = df.drop(columns=["orig_zone_name_short","dest_zone_name_short"], errors="ignore")

df = df.merge(zone_lookup.add_prefix("orig_"), left_on="dms_orig", right_on="orig_zone_id", how="left")
df = df.merge(zone_lookup.add_prefix("dest_"), left_on="dms_dest", right_on="dest_zone_id", how="left")
df = df.drop(columns=["orig_zone_id", "dest_zone_id"], errors="ignore")

df[["dms_orig","orig_zone_name_short","dms_dest","dest_zone_name_short"]].head()


,dms_orig,orig_zone_name_short,dms_dest,dest_zone_name_short
0,11.0,Birmingham AL,11.0,Birmingham AL
1,11.0,Birmingham AL,19.0,Rest of AL
2,11.0,Birmingham AL,129.0,Rest of FL
3,11.0,Birmingham AL,131.0,Atlanta GA
4,11.0,Birmingham AL,139.0,Rest of GA


In [8]:

# 6) Attach Metro GEOIDs + Centroids

if "FAF_Zone" not in metro_gdf.columns:
    raise ValueError("Expected 'FAF_Zone' column in metro shapefile. Please adjust column names.")

metro_gdf["FAF_Zone"] = pd.to_numeric(metro_gdf["FAF_Zone"], errors="coerce")

metro_proj = metro_gdf.to_crs(epsg=3857)
metro_proj["centroid"] = metro_proj.geometry.centroid
centroids_wgs = metro_proj["centroid"].to_crs(epsg=4326)

metro_gdf["centroid_lon"] = centroids_wgs.x
metro_gdf["centroid_lat"] = centroids_wgs.y

metro_lookup = metro_gdf[["FAF_Zone","GEOID","centroid_lon","centroid_lat"]].copy()

# Prevent *_x/*_y chaos if this notebook cell is re-run:
geo_cols = [c for c in df.columns if c.startswith("orig_GEOID") or c.startswith("dest_GEOID")
            or c.startswith("orig_centroid_") or c.startswith("dest_centroid_")
            or c in ["orig_FAF_Zone","dest_FAF_Zone"]]
df = df.drop(columns=geo_cols, errors="ignore")

df = df.merge(metro_lookup.add_prefix("orig_"), left_on="dms_orig", right_on="orig_FAF_Zone", how="left")
df = df.merge(metro_lookup.add_prefix("dest_"), left_on="dms_dest", right_on="dest_FAF_Zone", how="left")
df = df.drop(columns=["orig_FAF_Zone","dest_FAF_Zone"], errors="ignore")

# normalize:
for base in ["orig_GEOID","orig_centroid_lon","orig_centroid_lat","dest_GEOID","dest_centroid_lon","dest_centroid_lat"]:
    if base not in df.columns:
        if f"{base}_x" in df.columns:
            df[base] = df[f"{base}_x"]
        elif f"{base}_y" in df.columns:
            df[base] = df[f"{base}_y"]

need_geo = ["orig_GEOID","orig_centroid_lon","orig_centroid_lat","dest_GEOID","dest_centroid_lon","dest_centroid_lat"]
missing_geo = [c for c in need_geo if c not in df.columns]
if missing_geo:
    raise ValueError(f"Metro merge did not produce expected columns: {missing_geo}")

df[["dms_orig","orig_GEOID","orig_centroid_lon","orig_centroid_lat","dms_dest","dest_GEOID","dest_centroid_lon","dest_centroid_lat"]].head()


,dms_orig,orig_GEOID,orig_centroid_lon,orig_centroid_lat,dms_dest,dest_GEOID,dest_centroid_lon,dest_centroid_lat
0,11.0,01009,-86.622215,33.423699,11.0,01009,-86.622215,33.423699
1,11.0,01009,-86.622215,33.423699,19.0,01027,-86.793479,32.828845
2,11.0,01009,-86.622215,33.423699,129.0,12129,-83.153746,28.751312
3,11.0,01009,-86.622215,33.423699,131.0,13171,-84.307833,33.739472
4,11.0,01009,-86.622215,33.423699,139.0,13189,-83.330001,32.399567


In [9]:
# ============================================================
# 7) ArcGIS-ready trimming BEFORE scenario expansion
# ============================================================



arcgis_cols = [
    # IDs / categories
    "id",
    "dms_orig",
    "dms_dest",
    "dms_mode",
    "mode_name",
    "mode_pred",
    "sctg2",
    "trade_type",
    "dist_band",

    # shipment scale
    "tons",
    "value_2024_musd",
    "value_per_ton",
    "ton_miles",
    "distance_miles",
    "distance_km",
    "ton_km",

    # baseline totals
    "cost_estimate",
    "co2_tonnes",

    # RF predictions
    "cost_pred",
    "co2_pred_tonnes",
    "gen_cost_pred",

    # KAN predictions
    "cost_pred_kan",
    "co2_pred_tonnes_kan",

    # zone labels
    "orig_zone_name_short",
    "dest_zone_name_short",

    # ArcGIS geometry helper fields
    "orig_GEOID",
    "orig_centroid_lon",
    "orig_centroid_lat",
    "dest_GEOID",
    "dest_centroid_lon",
    "dest_centroid_lat",
]


arcgis_cols = [c for c in arcgis_cols if c in df.columns]

df_arc = df[arcgis_cols].copy()


numeric_cols = [
    "dms_orig", "dms_dest", "dms_mode", "tons",
    "value_2024_musd", "value_per_ton",
    "ton_miles", "distance_miles", "distance_km", "ton_km",
    "cost_estimate", "co2_tonnes",
    "cost_pred", "co2_pred_tonnes", "gen_cost_pred",
    "cost_pred_kan", "co2_pred_tonnes_kan",
    "orig_centroid_lon", "orig_centroid_lat",
    "dest_centroid_lon", "dest_centroid_lat",
]

for c in numeric_cols:
    if c in df_arc.columns:
        df_arc[c] = pd.to_numeric(df_arc[c], errors="coerce")


coord_cols = [
    "orig_centroid_lon",
    "orig_centroid_lat",
    "dest_centroid_lon",
    "dest_centroid_lat",
]

df_arc = df_arc.dropna(subset=coord_cols)


df_arc = df_arc[df_arc["dms_orig"] != df_arc["dms_dest"]]

print("ArcGIS base table shape before sampling:", df_arc.shape)
print("Columns kept:", df_arc.columns.tolist())

ArcGIS base table shape before sampling: (1638301, 29)
Columns kept: ['id', 'dms_orig', 'dms_dest', 'dms_mode', 'mode_name', 'mode_pred', 'sctg2', 'trade_type', 'dist_band', 'tons', 'value_2024_musd', 'ton_miles', 'distance_miles', 'ton_km', 'cost_estimate', 'co2_tonnes', 'cost_pred', 'co2_pred_tonnes', 'gen_cost_pred', 'cost_pred_kan', 'co2_pred_tonnes_kan', 'orig_zone_name_short', 'dest_zone_name_short', 'orig_GEOID', 'orig_centroid_lon', 'orig_centroid_lat', 'dest_GEOID', 'dest_centroid_lon', 'dest_centroid_lat']


In [10]:
# ============================================================
# 8) Sample BEFORE scenario expansion
# ============================================================

N_OUT = 10_000


scenario_multipliers_m = [1.0, 1.5]
scenario_multipliers_c = [1.0, 1.5]

n_scenarios = len(scenario_multipliers_m) * len(scenario_multipliers_c)
n_base_sample = max(1, N_OUT // n_scenarios)

if len(df_arc) > n_base_sample:
    df_arc_sample = df_arc.sample(n=n_base_sample, random_state=42).copy()
else:
    df_arc_sample = df_arc.copy()

print("Sampled base OD rows:", df_arc_sample.shape)
print("Expected final scenario rows:", len(df_arc_sample) * n_scenarios)

Sampled base OD rows: (2500, 29)
Expected final scenario rows: 10000


In [11]:

# ============================================================
# 9) Scenario expansion on the SMALL sampled table
# ============================================================

scenario_rows = []

for m in scenario_multipliers_m:
    for c in scenario_multipliers_c:
        tmp = df_arc_sample.copy()

        tmp["fuel_price_hike"] = m
        tmp["stricter_emissions_regulations"] = c
        tmp["scenario_label"] = f"{m:.1f}x fuel price, {c:.1f}x emissions"

        # Baseline scenario-adjusted values
        tmp["baseline_cost_scenario"] = tmp["cost_estimate"] * m
        tmp["baseline_co2_scenario"] = tmp["co2_tonnes"] * c

        # RF scenario-adjusted values
        tmp["rf_cost_scenario"] = tmp["cost_pred"] * m
        tmp["rf_co2_scenario"] = tmp["co2_pred_tonnes"] * c
        tmp["rf_generalized_cost_scenario"] = (
            tmp["rf_cost_scenario"] + tmp["rf_co2_scenario"]
        )

        # KAN scenario-adjusted values
        tmp["kan_cost_scenario"] = tmp["cost_pred_kan"] * m
        tmp["kan_co2_scenario"] = tmp["co2_pred_tonnes_kan"] * c
        tmp["kan_generalized_cost_scenario"] = (
            tmp["kan_cost_scenario"] + tmp["kan_co2_scenario"]
        )

     
        tmp["rf_cost_delta"] = tmp["rf_cost_scenario"] - tmp["baseline_cost_scenario"]
        tmp["rf_co2_delta"] = tmp["rf_co2_scenario"] - tmp["baseline_co2_scenario"]

        tmp["kan_cost_delta"] = tmp["kan_cost_scenario"] - tmp["baseline_cost_scenario"]
        tmp["kan_co2_delta"] = tmp["kan_co2_scenario"] - tmp["baseline_co2_scenario"]

        scenario_rows.append(tmp)

df_final = pd.concat(scenario_rows, ignore_index=True)

print("Final ArcGIS scenario table shape:", df_final.shape)
df_final.head()

Final ArcGIS scenario table shape: (10000, 44)


,id,dms_orig,dms_dest,dms_mode,mode_name,mode_pred,sctg2,trade_type,dist_band,tons,...,rf_cost_scenario,rf_co2_scenario,rf_generalized_cost_scenario,kan_cost_scenario,kan_co2_scenario,kan_generalized_cost_scenario,rf_cost_delta,rf_co2_delta,kan_cost_delta,kan_co2_delta
0,697420,19.0,131.0,1.0,truck,truck,24.0,2.0,2.0,13.473,...,674.39355,0.374966,674.768516,1053.3002,0.385303,1053.685503,-100.517924,0.005114,278.388726,0.015451
1,174974,171.0,20.0,4.0,air,multi,27.0,1.0,8.0,14.393,...,35458.07000,6.948848,35465.018848,15693.3240,5.428559,15698.752559,-175733.675025,-30.726841,-195498.421025,-32.247130
2,865283,486.0,181.0,5.0,multi,truck,33.0,2.0,6.0,486.911,...,59120.08200,36.460934,59156.542934,175980.2300,69.312744,176049.542744,-2322.633186,12.111348,114537.514814,44.963158
3,1060738,341.0,19.0,3.0,water,truck,39.0,2.0,8.0,122.881,...,45975.86700,22.563194,45998.430194,71790.3700,31.816368,71822.186368,42890.679457,17.064663,68705.182457,26.317837
4,422739,341.0,409.0,1.0,truck,truck,38.0,1.0,7.0,5.680,...,3548.21460,1.398587,3549.613187,2099.7517,1.000868,2100.752568,1811.129308,0.065519,362.666408,-0.332200


In [12]:

# ============================================================
# 10) Final ArcGIS column trim
# ============================================================



final_arcgis_cols = [

    "id",
    "scenario_label",
    "fuel_price_hike",
    "stricter_emissions_regulations",

    # OD info
    "dms_orig",
    "dms_dest",
    "orig_zone_name_short",
    "dest_zone_name_short",
    "orig_GEOID",
    "dest_GEOID",

    # freight attributes
    "dms_mode",
    "mode_name",
    "mode_pred",
    "sctg2",
    "trade_type",
    "dist_band",
    "tons",
    "value_2024_musd",
    "distance_miles",
    "distance_km",
    "ton_km",

    # baseline scenario values
    "baseline_cost_scenario",
    "baseline_co2_scenario",

    # RF scenario values
    "rf_cost_scenario",
    "rf_co2_scenario",
    "rf_generalized_cost_scenario",
    "rf_cost_delta",
    "rf_co2_delta",

    # KAN scenario values
    "kan_cost_scenario",
    "kan_co2_scenario",
    "kan_generalized_cost_scenario",
    "kan_cost_delta",
    "kan_co2_delta",

    # coordinates
    "orig_centroid_lon",
    "orig_centroid_lat",
    "dest_centroid_lon",
    "dest_centroid_lat",
]

final_arcgis_cols = [c for c in final_arcgis_cols if c in df_final.columns]
df_final = df_final[final_arcgis_cols].copy()

print("Final trimmed columns:", len(df_final.columns))
print(df_final.columns.tolist())
print("Final rows:", len(df_final))

Final trimmed columns: 36
['id', 'scenario_label', 'fuel_price_hike', 'stricter_emissions_regulations', 'dms_orig', 'dms_dest', 'orig_zone_name_short', 'dest_zone_name_short', 'orig_GEOID', 'dest_GEOID', 'dms_mode', 'mode_name', 'mode_pred', 'sctg2', 'trade_type', 'dist_band', 'tons', 'value_2024_musd', 'distance_miles', 'ton_km', 'baseline_cost_scenario', 'baseline_co2_scenario', 'rf_cost_scenario', 'rf_co2_scenario', 'rf_generalized_cost_scenario', 'rf_cost_delta', 'rf_co2_delta', 'kan_cost_scenario', 'kan_co2_scenario', 'kan_generalized_cost_scenario', 'kan_cost_delta', 'kan_co2_delta', 'orig_centroid_lon', 'orig_centroid_lat', 'dest_centroid_lon', 'dest_centroid_lat']
Final rows: 10000


In [13]:
# ============================================================
# 11) Build OD Line GeoJSON
# ============================================================

OUTPUT_GEOJSON = Path("faf5_2024_scenarios_10k_OD_lines_lines.geojson")

df_out = df_final.copy()

df_out["geometry"] = [
    LineString([
        (float(o_lon), float(o_lat)),
        (float(d_lon), float(d_lat))
    ])
    for o_lon, o_lat, d_lon, d_lat in zip(
        df_out["orig_centroid_lon"],
        df_out["orig_centroid_lat"],
        df_out["dest_centroid_lon"],
        df_out["dest_centroid_lat"]
    )
]

gdf_out = gpd.GeoDataFrame(df_out, geometry="geometry", crs="EPSG:4326")

gdf_out.to_file(OUTPUT_GEOJSON, driver="GeoJSON")

print("Wrote:", OUTPUT_GEOJSON)
print("Features:", len(gdf_out))
print("Columns:", len(gdf_out.columns))

Wrote: faf5_2024_scenarios_10k_OD_lines_lines.geojson
Features: 10000
Columns: 37
